In [19]:
import json
import os
from pathlib import Path

from sklearn.metrics.pairwise import cosine_similarity

with open(Path.cwd().parent/"parsed_resumes.json") as file:
    parsed_resumes=json.load(file)

In [20]:
sample=parsed_resumes["2.pdf"].copy()

In [21]:
sample

{'name': 'Deep M. Mehta',
 'email': 'mail@deepmehta.co.in',
 'phone': None,
 'skills': ['C#',
  'Java',
  'Python',
  'Ruby',
  'Javascript',
  'SQL',
  'Angular',
  'React',
  'Android',
  'React-Native',
  'Django',
  'Azure',
  'AWS',
  'Firebase',
  'App-Service',
  'Functions',
  'AD',
  'KeyVault',
  'APIM',
  'CosmosDB',
  'Queue',
  'Redis',
  'IoT-Hub',
  'App-Gateway',
  'Load-Balancer',
  'VNet',
  'VPN',
  'Storage',
  'Cognitive-Services',
  'FHIR',
  'Azure-DevOps',
  'GitHub Actions',
  'Terraform',
  'ARM-Templates',
  'Ansible',
  'Kubernetes',
  'LLMs',
  'PowerShell',
  'Unix Shell'],
 'experience': [{'company': 'Microsoft',
   'role_keywords': ['Software Engineer'],
   'skills': ['Azure-DevOps',
    'GitHub Actions',
    'Terraform',
    'ARM-Templates',
    'Ansible',
    'Kubernetes'],
   'duration_months': None,
   'impact_keywords': ['Experiences + Devices organization']},
  {'company': 'Amazon',
   'role_keywords': ['Software Development Engineer Intern',
    '

In [50]:
def clean_file(file:dict):
    if "name" in file:
        del file["name"]
    if "email" in file:
        del file["email"]
    if "phone" in file:
        del file["phone"]
    return file
sample=clean_file(sample)
sample

{'skills': ['C#',
  'Java',
  'Python',
  'Ruby',
  'Javascript',
  'SQL',
  'Angular',
  'React',
  'Android',
  'React-Native',
  'Django',
  'Azure',
  'AWS',
  'Firebase',
  'App-Service',
  'Functions',
  'AD',
  'KeyVault',
  'APIM',
  'CosmosDB',
  'Queue',
  'Redis',
  'IoT-Hub',
  'App-Gateway',
  'Load-Balancer',
  'VNet',
  'VPN',
  'Storage',
  'Cognitive-Services',
  'FHIR',
  'Azure-DevOps',
  'GitHub Actions',
  'Terraform',
  'ARM-Templates',
  'Ansible',
  'Kubernetes',
  'LLMs',
  'PowerShell',
  'Unix Shell'],
 'experience': [{'company': 'Microsoft',
   'role_keywords': ['Software Engineer'],
   'skills': ['Azure-DevOps',
    'GitHub Actions',
    'Terraform',
    'ARM-Templates',
    'Ansible',
    'Kubernetes'],
   'duration_months': None,
   'impact_keywords': ['Experiences + Devices organization']},
  {'company': 'Amazon',
   'role_keywords': ['Software Development Engineer Intern',
    'Full stack',
    'Mobile App',
    'Backend APIs',
    'Cloud'],
   'skills'

In [23]:
def extract_words_from_file(file):
    my_list=[]
    if file is None:
        return my_list
    if isinstance(file,str):
        return file.lower().split()
    elif isinstance(file,dict):
        for value in file.values():
            my_list.extend(extract_words_from_file(value))
        return my_list
    elif isinstance(file,list):
        for value in file:
            my_list.extend(extract_words_from_file(value))
        return my_list
    return my_list
print(extract_words_from_file(sample))

['c#', 'java', 'python', 'ruby', 'javascript', 'sql', 'angular', 'react', 'android', 'react-native', 'django', 'azure', 'aws', 'firebase', 'app-service', 'functions', 'ad', 'keyvault', 'apim', 'cosmosdb', 'queue', 'redis', 'iot-hub', 'app-gateway', 'load-balancer', 'vnet', 'vpn', 'storage', 'cognitive-services', 'fhir', 'azure-devops', 'github', 'actions', 'terraform', 'arm-templates', 'ansible', 'kubernetes', 'llms', 'powershell', 'unix', 'shell', 'microsoft', 'software', 'engineer', 'azure-devops', 'github', 'actions', 'terraform', 'arm-templates', 'ansible', 'kubernetes', 'experiences', '+', 'devices', 'organization', 'amazon', 'software', 'development', 'engineer', 'intern', 'full', 'stack', 'mobile', 'app', 'backend', 'apis', 'cloud', 'react', 'native', 'java', 'aws', 'textract', 'ocr', '50k_users', '85_percent_efficiency', 'automation', 'microsoft', 'technology', 'consultant', 'full', 'stack', 'web', 'apps', 'backend', 'apis', 'cloud', 'iot', 'cloud', 'governance', 'devops', 'inf

In [24]:
import xxhash
def word_frequency_map_for_file(file):
    map={}
    word_list=extract_words_from_file(file)
    for word in word_list:
        temp=xxhash.xxh64(word).intdigest()
        if temp in map.keys():
            map[temp]+=1
        else:
            map[temp]=1
    return map

In [25]:
def total_words_in_map(map:dict):
    return sum(value for value in map.values())

In [26]:
def word_tf(word:str,map:dict):
    temp=xxhash.xxh64(word).intdigest()
    if temp not in map:
        return 0
    return map[temp]/total_words_in_map(map)
print(word_tf("sql",word_frequency_map_for_file(sample)))

0.00625


In [27]:
def file_tf(sample:dict):
    sample_tf={}
    sample_map=word_frequency_map_for_file(sample)
    sample_words=extract_words_from_file(sample)
    for word in sample_words:
        sample_tf[xxhash.xxh64(word).intdigest()]=word_tf(word,sample_map)
    return sample_tf
sample_tf=file_tf(sample)

In [28]:
new_parsed_resumes={}
all_maps_dict={}
for key, value in parsed_resumes.items():
    new_parsed_resumes[key]=clean_file(value)
for key, value in new_parsed_resumes.items():
    all_maps_dict[key]=word_frequency_map_for_file(value)

In [29]:
def file_df(sample:dict,all_maps_dict):
    sample_map=word_frequency_map_for_file(sample)
    sample_df={k: 0 for k in sample_map.keys()}
    for file in all_maps_dict:
        for key in all_maps_dict[file]:
            if key in sample_map:
                sample_df[key]+=1
    return sample_df
sample_df=file_df(sample,all_maps_dict)

In [30]:
import math
total_resumes=18
def file_idf(sample:dict,total_resumes:int,all_maps_dict:dict):
    sample_df=file_df(sample,all_maps_dict)
    sample_idf={k:(math.log((1+total_resumes)/(1+x)) +1) for k,x in sample_df.items()}
    return sample_idf
sample_idf=file_idf(sample,total_resumes,all_maps_dict)

In [31]:
def file_tfidf(sample:dict,total_resumes:dict,all_maps_dict:dict):
    sample_tf=file_tf(sample)
    sample_idf=file_idf(sample,total_resumes,all_maps_dict)
    return {k:sample_tf[k]*sample_idf[k] for k in sample_tf.keys()}
sample_tfidf=file_tfidf(sample,total_resumes,all_maps_dict)

In [34]:
words=extract_words_from_file(sample)
temp={}
for word in words:
    temp[word]=sample_tfidf[xxhash.xxh64(word).intdigest()]
temp=dict(sorted(temp.items(), key=lambda x:x[1], reverse=True))

In [57]:
job_description = ["python java c++ javascript data structures algorithms backend development rest apis microservices sql postgresql mysql redis mongodb aws azure gcp docker kubernetes terraform ci cd github git system design distributed systems testing unit integration scalability performance optimization"]
job_description={"description": extract_words_from_file(job_description)}

In [67]:
all_file_df={}
for file in all_maps_dict.keys():
    for key in all_maps_dict[file].keys():
        all_file_df[key]=all_file_df.get(key,0)+1

In [69]:
all_file_idf={k:(math.log((1+total_resumes)/(1+v)) +1) for k,v in all_file_df.items()}

In [75]:
job_description_tf=file_tf(job_description)
job_description_tfidf={k:job_description_tf[k]*(all_file_idf.get(k,0)) for k,v in job_description_tf.items()}

In [81]:
all_file_tfidf={k: file_tfidf(new_parsed_resumes[k],total_resumes,all_maps_dict) for k in all_maps_dict.keys()}

In [80]:
cosine_similarity={file:sum(job_description_tfidf[k]*all_file_tfidf[file].get(k,0) for k in job_description_tfidf)/((math.sqrt(sum(v*v for v in all_file_tfidf[file].values())))*(math.sqrt(sum(v*v for v in job_description_tfidf.values())))) for file in all_file_tfidf}
cosine_similarity

{'2.pdf': 0.32536205723889616,
 '3.pdf': 0.1281038421612027,
 '4.pdf': 0.08446340734484044,
 '5.pdf': 0.10508500131647466,
 '6.pdf': 0.12896476058511017,
 '7.pdf': 0.0,
 '8.pdf': 0.0,
 '9.pdf': 0.03812431352785597,
 '10.pdf': 0.009670104149897205,
 '11.pdf': 0.0,
 '12.pdf': 0.027370797960997412,
 '13.pdf': 0.011792765785555184,
 '14.pdf': 0.009379862966508017,
 '15.pdf': 0.03997584155245943,
 '16.pdf': 0.13038919739164578,
 '17.pdf': 0.09066398435391439,
 '18.pdf': 0.09724086128719302,
 '19.pdf': 0.1370426234651992}